# Delegate to a subagent

Give an auditor its own instructions, model, and restricted tool selection.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/03_subagents.ipynb)

Run these cells in order in **Google Colab**. Everything runs in its cloud runtime:
no repository checkout or laptop installation. A CPU runtime is enough.
Model calls use your provider account. Clear outputs before sharing a saved copy.

## Install and choose a model

Keep the defaults for a first run. To switch providers, change `MODEL`:
- OpenAI: `openai/gpt-5.4-mini` with `OPENAI_API_KEY`.
- Anthropic: `anthropic/claude-sonnet-4-6` with `ANTHROPIC_API_KEY`.
- OpenRouter: `openrouter/anthropic/claude-sonnet-4.6` with `OPENROUTER_API_KEY`.

LiteLLM's Python SDK handles provider translation; no gateway is required.
Optional: set `API_BASE` to your gateway URL and use its exact model alias
with `LITELLM_API_KEY`. Leave `API_BASE` empty for direct provider access.

For Gemini, Groq, Mistral, DeepSeek, Together AI, xAI, Azure, Bedrock,
Vertex AI, or Ollama, see the [model setup guide](https://github.com/BerriAI/liteagents/blob/main/docs/models.md).
Cloud authentication and provider-specific environment variables must be configured
in this runtime before running the agent.

In [ ]:
import os

HARNESS = os.environ.get("LITEAGENTS_HARNESS", "deepagents")
MODEL = os.environ.get("LITEAGENTS_MODEL", "openai/gpt-5.4-mini")
API_BASE = os.environ.get("LITEAGENTS_API_BASE", "")  # Optional gateway URL.

Available harnesses: `deepagents`, `pydantic-ai`, `claude-sdk`, `codex`,
`opencode-v1`, `opencode-v2`. Rerun the install cell after changing your selection.
Packages are reused within this runtime; a fresh Colab runtime needs its own install.
OpenCode is installed only when selected.

This preview installs from a GitHub release wheel because the PyPI name currently
belongs to another package. It does not clone the repository.

In [ ]:
# @title Install selected integrations
import shutil
import subprocess
import sys

selected_harnesses = [HARNESS]
extras = sorted(set(selected_harnesses))
release = "https://github.com/BerriAI/liteagents/releases/download/v0.3.0a3"
package = f"liteagents[{','.join(extras)}] @ {release}/liteagents-0.3.0a3-py3-none-any.whl"
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", package,
    "-c", f"{release}/constraints-tested.txt",
])
if any(h.startswith("opencode-") for h in selected_harnesses):
    if shutil.which("opencode") is None:
        subprocess.check_call(["npm", "install", "-g", "opencode-ai@1.18.29"])
    subprocess.check_call(["opencode", "--version"])
print("Ready:", ", ".join(selected_harnesses))

## Add your API key

In Colab, open the **key icon → Secrets**, add the key named above, and enable
notebook access. Or enter it in the hidden prompt below. The key stays out of your
code and saved outputs. If installation asks for a runtime restart,
restart once and run the cells again.

In [ ]:
# @title Connect your provider
from getpass import getpass

KEY_NAME = "LITELLM_API_KEY" if API_BASE else {
    "openai": "OPENAI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
    "gemini": "GEMINI_API_KEY",
    "groq": "GROQ_API_KEY",
    "mistral": "MISTRAL_API_KEY",
    "together_ai": "TOGETHERAI_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "xai": "XAI_API_KEY",
    "azure": "AZURE_API_KEY",
}.get(MODEL.split("/", 1)[0])
API_KEY = None
if KEY_NAME:
    API_KEY = os.environ.get(KEY_NAME)
    if not API_KEY:
        try:
            from google.colab import userdata
        except ImportError:
            pass
        else:
            try:
                API_KEY = userdata.get(KEY_NAME)
            except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
                pass
    API_KEY = API_KEY or getpass(f"{KEY_NAME}: ")
    if not API_KEY:
        raise ValueError(f"Provide {KEY_NAME} before running the agent.")
    os.environ[KEY_NAME] = API_KEY
else:
    print("Using provider credentials from the runtime; see the model setup guide.")
os.environ["LITEAGENTS_MODEL"] = MODEL
if API_BASE:
    os.environ["LITEAGENTS_API_BASE"] = API_BASE
MODEL_KWARGS = {"api_base": API_BASE, "api_key": API_KEY} if API_BASE else {}

## Create your profile

This is the SDK interface. The following cells change this profile to demonstrate
one feature. A temporary workspace keeps each run's files separate.

In [ ]:
import asyncio
import tempfile
from pathlib import Path

from liteagents import LiteAgentClient, LiteAgentOptions, ProfileOptions

profile = ProfileOptions(
    harness=HARNESS,
    model=MODEL,
    model_kwargs=MODEL_KWARGS,
    tools=[],
    max_turns=10,
    system_prompt="Use the requested tools and report their actual results. Be concise.",
)
workspace_root = Path(os.environ.get(
    "LITEAGENTS_NOTEBOOK_WORKSPACE", Path(tempfile.gettempdir()) / "liteagents-notebooks"
))
workspace_root.mkdir(parents=True, exist_ok=True)
workspace = Path(tempfile.mkdtemp(prefix="run-", dir=workspace_root)).resolve()
print("Workspace:", workspace)

In [ ]:
CHILD_MODEL = MODEL  # Or another model from the same provider.
PROMPT = "Use delegate_auditor to verify the payment and total for order A123. Relay its findings."

In [ ]:
from liteagents import SubagentOptions

(workspace / "receipt.txt").write_text("Order A123 was paid. The total is USD 12.\n")
profile.tools = ["delegate_auditor"]
profile.features.subagents = True
profile.subagents = {"auditor": SubagentOptions(
    description="Verify orders against receipt.txt; report the evidence.",
    model=CHILD_MODEL,
    tools=["read_file"],
)}

## Watch the delegation

The parent has the delegation tool; the child can read the receipt. The child cannot edit files.

In [ ]:
delegation_events = []
async with asyncio.timeout(180):
    async with LiteAgentClient(options=LiteAgentOptions(profile=profile, cwd=workspace)) as client:
        run = await client.start_run(PROMPT)
        async for event in run.events():
            if event.kind in ("subagent_started", "subagent_completed"):
                delegation_events.append(event.kind)
                print(event.kind, event.data["agent"])
        result = await run.result()
print(result.text)
assert "subagent_completed" in delegation_events